# レッスン08 - マルチエージェントデザインパターン


## セットアップ


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## なぜマルチエージェントシステムか？

旅行の計画のような現実のタスクは、物流、地域の知識、予算管理など、多くの異なる専門知識を含んでいます。すべてを一人のエージェントが処理しようとすると、すぐに手に負えなくなります。

マルチエージェントシステムはこれを<strong>専門化</strong>で解決します：各エージェントが一つの専門分野に集中し、ジェネラリストよりも高品質な結果を生み出します。また、<strong>スケーラビリティ</strong>も向上します — 新しいエージェント（例：航空専門家、レストラン評論家）を既存のワークフローを書き換えることなく追加できます。エージェントは構造化されたパイプラインを通じて連携し、文脈を次々に渡します。


## 専門エージェントの作成


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 連続的なワークフローの構築

`WorkflowBuilder` を使うと、エージェントを有向グラフに接続できます。ここでは簡単な2ステップのパイプラインを作成します：**TravelPlanner** が旅程の草案を作成し、その後 **TravelConcierge** がそれを確認し、強化します。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ワークフローにエージェントを追加する

マルチエージェントパターンの最大の利点の一つは、その拡張のしやすさです。以下では、旅行者の予算と計画を照らし合わせて、コストが限度を超えそうな項目にフラグを立て、節約代替案を提案する **BudgetReviewer** エージェントを追加します。ワークフローは現在、3つのエージェントを順番に実行します：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## サマリー

このレッスンで学んだこと：

1. <strong>専門的なエージェントの作成</strong> — 計画、コンシェルジュ、予算審査など、特定の役割を持つエージェントを作成。
2. **`WorkflowBuilder` と `add_edge` を使ってエージェントを順次のワークフローに連結。**
3. **マルチエージェントパイプラインからの出力をストリームし、どのエージェントが発話しているかを追跡。**
4. **既存のエージェントを変更せずに、新しいエージェントをチェーンに追加してワークフローを拡張。**

マルチエージェント設計パターンは、各エージェントをシンプルに保ちつつ、単一のエージェントでは達成できない、より豊かで綿密なレビュー結果を生み出します。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
